This is a follow-up example showing how to deal with transition metal ions in the system, which may exhibit variable oxidation states. 

In [2]:
from py_oats import OnsagerTransportAnalyzer, plot_correlation_pairwise, plot_all_correlations
from py_oats.utils.decorators import CHGNetChargeDecorator, BVChargeDecorator
import numpy as np
import matplotlib.pyplot as plt

If your trajectory consists of structures that have been decorated with the correct oxidation state, just pass them as is to the analyzer object and the analysis will account for the oxidation states automatically. 

However, if that is not the case, I've found empirically (esp. for amorphous solids) that oxidation state assignment of transition metal ions is most stable when done using their magnetic moments.

Hence, the preferred approach is to do a DFT static on atleast one frame of your trajectory to obtain the site-wise magnetic moments and use those to assign oxidation states. If your system is too big for a static, or you expect there to be charge transfer during your simulation, a cheaper way is to use a pretrained foundational CHGNet model to compute the site-wise magnetic moments. Functionality to do this is present in the `CHGNetChargeDecorator` class. If you do not trust CHGNet, and want a more intuive approach, try the `BVChargeDecorator`, which uses the bond valence analyzer from `pymatgen.analysis.bond_valence` to assign oxidation states. Keep in mind though that using bond valences has been shown to fail when dealing structures far away from equilibrium crystals. 

In [7]:
trajectory_path = './LiMn2O4-1250K.dump'

Regardless of the method of choice, we first need to specify the range of oxidation states as a dict for all species that we expect in the system for the analyzer. In this case, we'll be analyzing a trajectory of amorphous LiMn2O4, which has mixed +3 and +4 Mn. If you do not know this a-priori, running a bond valence analysis might be of help.

In [4]:
oxidation_states = {"Li" : [1], "Mn" : [3, 4], "O" : [-2]}

If you do not specify a decorator object, the analyzer class defaults to the `Structure.guess_oxidation_state_by_element()` function, which may not be the result you are looking for. Here, we'll use the CHGNet-based decorator.

In [5]:
model_path = None #'/path/to/pretrained/chgnet.pth.tar'
decorator = CHGNetChargeDecorator(model_path=model_path, 
                                  oxidation_states=oxidation_states,
                                  adaptive_boundaries=True)

CHGNet v0.3.0 initialized with 412,525 parameters
CHGNet will run on mps


If you have a fine-tuned CHGNet for your system, you can pass in the path to the model, else the decorator defaults to using the latest MPTrj version of CHGNet. Turning on `adaptive_boundaries` searches for the optimal bounds using linear programming for the specified oxidation states that best ensure charge neutrality of the trajectory. This is good to have turned on, but keep in mind that the bound that is found is heavily dependent on how well CHGNet describes the magnetic moment surface for your system, and this can improved significantly by fine-tuning a CHGNet before doing this analysis.

In [8]:
analyzer = OnsagerTransportAnalyzer.from_lammps_dump(dump_file=trajectory_path, 
                                                     charge_decorator=decorator,
                                                     temperature=1250,
                                                     species=['Li', 'Mn', 'O'],
                                                     oxidation_states={'Li': [1], 'Mn': [3, 4], 'O': [-2]},
                                                     decorate_freq=-1,
                                                     step_skip=1000,
                                                     )

/Users/virkaran/miniconda3/envs/charged_oats/lib/python3.10/site-packages/chgnet/model/model.py:889: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  volumes = torch.tensor(volumes, dtype=TORCH_DTYPE, device=atomic_numbers.device)


Got the decorated structures!


/Users/virkaran/Code/Polished/py-OATS/src/py_oats/utils/fitting.py:26: RuntimeWarning: invalid value encountered in log
  slope, intercept, r_value, p_value, std_err = linregress(np.log(times[intervals[i]: intervals[i+1]]), np.log(f[intervals[i]:intervals[i+1]]))


In [11]:
analyzer.species

['Li+1', 'Mn+3', 'Mn+4', 'O-2']

With that, we can now see the analyzer has split the Mn into 2 distinct specie, and the computed transport tensors are 4x4 instead of 3x3!

In [9]:
analyzer.L_tensor

array([[ 5.21907810e+17,  2.38305234e+15, -3.00744837e+16,
        -5.53536407e+16],
       [ 2.38305234e+15,  3.14853698e+15, -2.60784261e+15,
         1.08344020e+15],
       [-3.00744837e+16, -2.60784261e+15,  3.55174312e+15,
         1.88045403e+15],
       [-5.53536407e+16,  1.08344020e+15,  1.88045403e+15,
         5.93577887e+15]])

We can also observe an interesting phenomenon from this tensor: in this trajectory, Li is positively correlated with Mn+3 and negatively with Mn+4, and there is an order of magnitude difference in the two, which has fascinating implications for the type of Li transport this system will show!